# Sprint 8 - Dual-head model with domain routing

**Why this sprint:** Sprint 7 proved the lab-vs-field trade-off is structural across 3
backbones. No single shared head can stay good at both domains. The fix: two independent
heads on the same backbone, each trained on its own domain, with a lightweight domain
classifier that routes each image to the correct head.

**Architecture:**
```
backbone (shared, frozen)
  ├── head_lab          -> trained on PlantVillage only
  ├── head_field        -> trained on PlantDoc only
  └── domain_classifier -> lab vs field (routes to correct head)
```

**Inference:** domain classifier decides which head to use per sample. No confidence race.

**Sprint 8 gates (must BOTH pass):**
1. **Field:** head_field on PlantDoc F1 >= 0.60 (stretch 0.70)
2. **Lab:** head_lab on PlantVillage F1 >= 0.85

### Where we are (Sprint 7 results)
- baseline (MobileNetV2): PlantVillage 0.9501 F1 | PlantDoc 0.1116 F1
- `both_resnet50` (v11): PlantDoc **0.6554** F1 but forgot lab (0.2857)
- `mixed` (MobileNetV2): PlantVillage **0.9592** F1 | PlantDoc 0.4107 F1
- `mixed_from_field_resnet50_x8` (v13_x8): PlantVillage 0.9508 F1 | PlantDoc 0.4709 F1

**The structural finding:** no single shared head clears both gates.
Dual-head + domain routing is the architectural fix.

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")
LOCAL_RAW_DIR = Path("/content/folium_raw")
LOCAL_DATA_DIR = Path("/content/folium_data")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

def run(cmd, cwd, label, stream=False):
    """Run a subprocess. If stream=True, print stdout line-by-line in real time."""
    if stream:
        proc = subprocess.Popen(
            cmd, cwd=str(cwd),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        lines = []
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        proc.wait()
        combined = "".join(lines)
        if proc.returncode != 0:
            print(f"\n[{label}] failed (returncode {proc.returncode})")
        assert proc.returncode == 0, label
        class _Result:
            pass
        r = _Result()
        r.stdout = combined
        r.stderr = ""
        r.returncode = 0
        return r
    result = subprocess.run(cmd, cwd=str(cwd), capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[{label}] failed (returncode {result.returncode})")
        print("stdout tail:\n", result.stdout[-2000:])
        print("stderr tail:\n", result.stderr[-2000:])
    assert result.returncode == 0, label
    return result

## Step 2 - Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn

## Step 2b - Clean old dual-head rows from ablation CSV

In [ ]:
import pandas as pd

csv_path = RESULTS_DIR / "ablation_results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    old_dual = df["variant"].str.startswith("dual_head") | df["variant"].str.startswith("s8_")
    n_old = old_dual.sum()
    if n_old > 0:
        df = df[~old_dual].reset_index(drop=True)
        df.to_csv(csv_path, index=False)
        print(f"Removed {n_old} old dual-head rows from {csv_path}")
    else:
        print("No old dual-head rows found.")
else:
    print("No ablation CSV yet.")

## Step 3 - Hydrate raw from Drive, then organize splits locally

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--seed", "42",
    "--val-fraction", "0.15",
    "--test-fraction", "0.15",
], cwd=str(REPO_DIR), label="organize_datasets.py failed")
print("Data ready at", LOCAL_DATA_DIR)

## Step 4 - Train head_lab on PlantVillage

Train only the lab head. Backbone + field head stay frozen.

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--backbone", "resnet50",
    "--epochs", "5",
    "--lr", "1e-3",
    "--augment",
    "--dual-head",
    "--train-head", "lab",
    "--tag", "dual_head_lab",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="dual-head lab training failed", stream=True)
print("\n=== Lab head training complete ===")

## Step 5 - Train head_field on PlantDoc

Load the lab checkpoint. Copy head_lab weights into head_field as warm start.
Freeze everything except head_field. Train on PlantDoc mapped to PV label space.

In [ ]:
DUAL_LAB_CKPT = CHECKPOINT_DIR / "best_plantvillage_dual_head_lab.pt"
assert DUAL_LAB_CKPT.exists(), f"Missing lab checkpoint: {DUAL_LAB_CKPT}. Run Step 4 first."

cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--map-to-pv",
    "--backbone", "resnet50",
    "--epochs", "10",
    "--lr", "1e-3",
    "--augment",
    "--dual-head",
    "--train-head", "field",
    "--init-from", str(DUAL_LAB_CKPT),
    "--tag", "dual_head_field",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="dual-head field training failed", stream=True)
print("\n=== Field head training complete ===")

## Step 6 - Train domain classifier

A single linear layer (2048 -> 2) that learns to distinguish lab photos from field photos.
Trained on PlantVillage (label=0, lab) + PlantDoc (label=1, field).
At inference, this routes each image to the correct head.

In [ ]:
DUAL_FIELD_CKPT = CHECKPOINT_DIR / "best_plantdoc_dual_head_field.pt"
assert DUAL_FIELD_CKPT.exists(), f"Missing dual-head checkpoint: {DUAL_FIELD_CKPT}. Run Step 5 first."

cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--backbone", "resnet50",
    "--epochs", "3",
    "--lr", "1e-3",
    "--augment",
    "--dual-head",
    "--train-head", "domain",
    "--init-from", str(DUAL_FIELD_CKPT),
    "--tag", "dual_head_domain",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
result = run(cmd, cwd=str(REPO_DIR), label="domain classifier training failed", stream=True)
print("\n=== Domain classifier training complete ===")

## Step 7 - Evaluate

Run 5 evaluations:
1. head_lab alone on PlantVillage (true lab score)
2. head_field alone on PlantDoc (true field score)
3. domain-routed on PlantVillage (production inference)
4. domain-routed on PlantDoc (production inference)
5. predict_dual on both (old confidence-race, for comparison)

In [ ]:
DOMAIN_CKPT = CHECKPOINT_DIR / "best_domain_dual_head_domain.pt"
assert DOMAIN_CKPT.exists(), f"Missing domain checkpoint: {DOMAIN_CKPT}. Run Step 6 first."

# 1) head_lab on PlantVillage
print("=" * 60)
print("7a) head_lab on PlantVillage (true lab score)")
print("=" * 60)
cmd = [
    sys.executable, "-m", "ml.evaluate",
    "--checkpoint", str(DOMAIN_CKPT),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--split", "test",
    "--dual-head",
    "--eval-head", "lab",
    "--variant", "s8_lab_on_plantvillage",
    "--results", str(RESULTS_DIR / "ablation_results.csv"),
]
result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

# 2) head_field on PlantDoc
print("\n" + "=" * 60)
print("7b) head_field on PlantDoc (true field score)")
print("=" * 60)
cmd = [
    sys.executable, "-m", "ml.evaluate",
    "--checkpoint", str(DOMAIN_CKPT),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--split", "test",
    "--map-to-pv",
    "--dual-head",
    "--eval-head", "field",
    "--variant", "s8_field_on_plantdoc",
    "--results", str(RESULTS_DIR / "ablation_results.csv"),
]
result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

# 3) domain-routed on PlantVillage
print("\n" + "=" * 60)
print("7c) domain-routed on PlantVillage")
print("=" * 60)
cmd = [
    sys.executable, "-m", "ml.evaluate",
    "--checkpoint", str(DOMAIN_CKPT),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--split", "test",
    "--dual-head",
    "--predict-mode", "routed",
    "--variant", "s8_routed_plantvillage",
    "--results", str(RESULTS_DIR / "ablation_results.csv"),
]
result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

# 4) domain-routed on PlantDoc
print("\n" + "=" * 60)
print("7d) domain-routed on PlantDoc")
print("=" * 60)
cmd = [
    sys.executable, "-m", "ml.evaluate",
    "--checkpoint", str(DOMAIN_CKPT),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--split", "test",
    "--map-to-pv",
    "--dual-head",
    "--predict-mode", "routed",
    "--variant", "s8_routed_plantdoc",
    "--results", str(RESULTS_DIR / "ablation_results.csv"),
]
result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

# 5) predict_dual on both (old confidence-race, for comparison)
print("\n" + "=" * 60)
print("7e) predict_dual on PlantVillage (confidence-race, for comparison)")
print("=" * 60)
cmd = [
    sys.executable, "-m", "ml.evaluate",
    "--checkpoint", str(DOMAIN_CKPT),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--split", "test",
    "--dual-head",
    "--predict-mode", "dual",
    "--variant", "s8_dual_plantvillage",
    "--results", str(RESULTS_DIR / "ablation_results.csv"),
]
result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

print("\n" + "=" * 60)
print("7f) predict_dual on PlantDoc (confidence-race, for comparison)")
print("=" * 60)
cmd = [
    sys.executable, "-m", "ml.evaluate",
    "--checkpoint", str(DOMAIN_CKPT),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--split", "test",
    "--map-to-pv",
    "--dual-head",
    "--predict-mode", "dual",
    "--variant", "s8_dual_plantdoc",
    "--results", str(RESULTS_DIR / "ablation_results.csv"),
]
result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

## Step 8 - The verdict

Per-head scores are ground truth. Routed scores show production performance.
Dual scores show the old confidence-race for comparison.

**Gates (per-head scores):**
- head_field on PlantDoc F1 >= 0.60
- head_lab on PlantVillage F1 >= 0.85

In [ ]:
import pandas as pd

df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()

print("=== Per-head scores (ground truth) ===")
per_head_keys = ["s8_lab_on_plantvillage", "s8_field_on_plantdoc"]
per_head = df[df["variant"].isin(per_head_keys)].copy()
if len(per_head) > 0:
    print(per_head[["variant", "f1"]].to_string(index=False))

print("\n=== Domain-routed scores (production) ===")
routed_keys = ["s8_routed_plantvillage", "s8_routed_plantdoc"]
routed = df[df["variant"].isin(routed_keys)].copy()
if len(routed) > 0:
    print(routed[["variant", "f1"]].to_string(index=False))

print("\n=== predict_dual scores (confidence-race, for comparison) ===")
dual_keys = ["s8_dual_plantvillage", "s8_dual_plantdoc"]
dual_rows = df[df["variant"].isin(dual_keys)].copy()
if len(dual_rows) > 0:
    print(dual_rows[["variant", "f1"]].to_string(index=False))

if len(per_head) == 2:
    field_f1 = per_head[per_head["variant"] == "s8_field_on_plantdoc"]["f1"].iloc[0]
    lab_f1 = per_head[per_head["variant"] == "s8_lab_on_plantvillage"]["f1"].iloc[0]
    field_pass = "PASS" if field_f1 >= 0.60 else "FAIL"
    lab_pass = "PASS" if lab_f1 >= 0.85 else "FAIL"
    print(f"\nVerdict: field {field_f1:.4f} -> {field_pass} | lab {lab_f1:.4f} -> {lab_pass}")
else:
    print("\nPer-head rows not found. Run Step 7 first.")

## Where things live

**On Google Drive:**
```
folium/checkpoints/best_plantvillage_dual_head_lab.pt     head_lab trained
folium/checkpoints/best_plantdoc_dual_head_field.pt       head_field trained (warm-started from lab)
folium/checkpoints/best_domain_dual_head_domain.pt        domain classifier trained
folium/results/ablation_results.csv                       all ablation rows
```